# Bước 1: Chuẩn Hóa Dữ Liệu & Xuất Tập Huấn Luyện (Data Scaling & Normalization)
Notebook này xử lý bài toán **khử độ lệch biên độ** giữa các loại cảm biến cho cả 2 tập dữ liệu **MIMIC-III** và **PTB-XL**.

Chúng ta thực hiện 2 kỹ thuật chuẩn hóa dữ liệu phổ biến:
1. **Z-Score Normalization (`StandardScaler`)**: Biến đổi dữ liệu có trung bình = 0 và độ lệch chuẩn = 1.
2. **Min-Max Scaling (`MinMaxScaler`)**: Biến đổi các đặc trưng về khoảng [0, 1].

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import os
import warnings
warnings.filterwarnings('ignore')

# Cấu hình font & giao diện biểu đồ
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (16, 8)

### 1. Nạp các tập dữ liệu đặc trưng từ `data/features/` (MIMIC-III & PTB-XL)

In [ ]:
data_dir_candidates = ['../../data/features', '../data/features', 'data/features']
data_dir = next((d for d in data_dir_candidates if os.path.exists(d)), None)
if not data_dir:
    raise FileNotFoundError("❌ Không tìm thấy thư mục data/features!")

datasets = {}

# 1. MIMIC-III
mimic_path = os.path.join(data_dir, 'mimic_features.csv')
if not os.path.exists(mimic_path):
    mimic_path = os.path.join(data_dir, 'mimic_train_features_advanced.csv')
if os.path.exists(mimic_path):
    df_mimic = pd.read_csv(mimic_path)
    datasets['MIMIC-III'] = df_mimic
    print(f"✅ Nạp MIMIC-III thành công: {df_mimic.shape[0]} mẫu, {df_mimic.shape[1]} cột")

# 2. PTB-XL
ptb_path = os.path.join(data_dir, 'ptbxl_features.csv')
if not os.path.exists(ptb_path):
    ptb_path = os.path.join(data_dir, 'ptbxl_features_advanced.csv')
if os.path.exists(ptb_path):
    df_ptb = pd.read_csv(ptb_path)
    datasets['PTB-XL'] = df_ptb
    print(f"✅ Nạp PTB-XL thành công: {df_ptb.shape[0]} mẫu, {df_ptb.shape[1]} cột")

for name, df in datasets.items():
    print(f"📌 {name}: Distribution = {dict(df['status'].value_counts())}")

### 2. Thực hiện Chuẩn hóa Z-Score & Min-Max Scaling cho từng tập dữ liệu

In [ ]:
scaled_results = {}

for name, df in datasets.items():
    X = df.drop(columns=['status'])
    y = df['status']
    feature_cols = X.columns.tolist()
    
    # Z-Score Normalization
    scaler_z = StandardScaler()
    X_z = pd.DataFrame(scaler_z.fit_transform(X), columns=feature_cols)
    df_zscore = pd.concat([X_z, y.reset_index(drop=True)], axis=1)
    
    # Min-Max Scaling
    scaler_mm = MinMaxScaler()
    X_mm = pd.DataFrame(scaler_mm.fit_transform(X), columns=feature_cols)
    df_minmax = pd.concat([X_mm, y.reset_index(drop=True)], axis=1)
    
    scaled_results[name] = {
        'raw': df,
        'zscore': df_zscore,
        'minmax': df_minmax
    }
    print(f"⚡ Đã chuẩn hóa Z-Score & Min-Max thành công cho: {name}")

### 3. Trực quan hóa phân bố SDNN trước và sau khi chuẩn hóa

In [ ]:
fig, axes = plt.subplots(len(datasets), 3, figsize=(18, 4.5 * len(datasets)))
if len(datasets) == 1:
    axes = [axes]

for i, (name, res) in enumerate(scaled_results.items()):
    sns.kdeplot(res['raw']['SDNN'], ax=axes[i][0], color='blue', fill=True)
    axes[i][0].set_title(f'{name} - SDNN Gốc (Original)', fontsize=13, fontweight='bold')
    
    sns.kdeplot(res['zscore']['SDNN'], ax=axes[i][1], color='green', fill=True)
    axes[i][1].set_title(f'{name} - Z-Score Scaled (Mean=0, Std=1)', fontsize=13, fontweight='bold')
    
    sns.kdeplot(res['minmax']['SDNN'], ax=axes[i][2], color='orange', fill=True)
    axes[i][2].set_title(f'{name} - Min-Max Scaled [0, 1]', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

### 4. Xuất dữ liệu đã chuẩn hóa vào thư mục `data/processed/`

In [ ]:
output_dir_candidates = ['../../data/processed', '../data/processed', 'data/processed']
output_dir = next((d for d in output_dir_candidates if os.path.exists(os.path.dirname(d))), '../../data/processed')
os.makedirs(output_dir, exist_ok=True)

for name, res in scaled_results.items():
    prefix = name.lower().replace('-', '').replace(' ', '_')
    z_path = os.path.join(output_dir, f'{prefix}_zscore_scaled.csv')
    mm_path = os.path.join(output_dir, f'{prefix}_minmax_scaled.csv')
    
    res['zscore'].to_csv(z_path, index=False)
    res['minmax'].to_csv(mm_path, index=False)
    
    print(f"🎉 [{name}] Z-Score Scaled -> {z_path}")
    print(f"🎉 [{name}] Min-Max Scaled -> {mm_path}")